# NPC Engine — LoRA fine-tune (Dream 7B)

Trains the NPC specialist on 74k dialogue triples (agentlans, CC-BY-4.0): persona + world state stay clean, only the NPC's line is masked — the same recipe the big diffusion labs use.

**Runtime must be a T4 GPU** (Runtime → Change runtime type → T4). Run the cells top to bottom.

## 1. Install (pinned — Dream's code breaks on newer transformers)

In [ ]:
!pip install -q transformers==4.53.3 peft accelerate bitsandbytes

## 2. Restore the npc_engine package

In [ ]:
# --- restore the npc_engine package (embedded in this notebook) ---
import base64, pathlib
FILES = {
    'npc_engine/__init__.py': 'IiIiTlBDIEVuZ2luZSBQaGFzZSAwIOKAlCBhIHRpbnkgdGV4dC1kaWZmdXNpb24gTlBDIGRpYWxvZ3VlIGhhcm5lc3MuCgpCdWlsdCBvbiBEcmVhbSA3QiBJbnN0cnVjdCAoRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCKSwgYSBtYXNrZWQgZGlmZnVzaW9uIExNLgpUaGUgZGlmZnVzaW9uLW5hdGl2ZSBmZWF0dXJlcyB1c2VkIGhlcmU6CgotIHBhcmFsbGVsIGNhbmRpZGF0ZSBnZW5lcmF0aW9uOiBOIG1hc2tlZCBjYW52YXNlcyBkZW5vaXNlZCBpbiBPTkUgYmF0Y2hlZCBwYXNzCiAgKEFSIHdvdWxkIG5lZWQgTiBzZXF1ZW50aWFsIGRlY29kZXMpCi0gY29uZmlkZW5jZSByYW5raW5nOiBwZXItdG9rZW4gbWF4LXByb2Igb3ZlciB0aGUgZ2VuZXJhdGVkIHNwYW4sIGZyb20gYSBzaW5nbGUKICBzY29yaW5nIGZvcndhcmQgcGFzcyDigJQgbm8gc2Vjb25kIG1vZGVsCi0gY29uZmlkZW5jZSBmYWxsYmFjazogaWYgdGhlIGJlc3QgY2FuZGlkYXRlIHNjb3JlcyBiZWxvdyB0aHJlc2hvbGQsIHRoZSBlbmdpbmUKICBzaGlwcyBhIHNhZmUgY2FubmVkIGxpbmUgaW5zdGVhZCBvZiBhIGJyb2tlbiBvbmUgKHRoZSBhbnRpLWdhbWVzbG9wIGd1YXJkKQoiIiIKCmZyb20gLnBlcnNvbmFzIGltcG9ydCBQZXJzb25hLCBTVEFSVEVSX1BFUlNPTkFTLCBzeXN0ZW1fcHJvbXB0X2Zvcgpmcm9tIC5lbmdpbmUgaW1wb3J0IE5QQ0VuZ2luZSwgTlBDUmVzcG9uc2UKCl9fdmVyc2lvbl9fID0gIjAuMi4yIgoKX19hbGxfXyA9IFsKICAgICJQZXJzb25hIiwgIlNUQVJURVJfUEVSU09OQVMiLCAic3lzdGVtX3Byb21wdF9mb3IiLAogICAgIk5QQ0VuZ2luZSIsICJOUENSZXNwb25zZSIsCl0K',
    'npc_engine/personas.py': 'IiIiUGVyc29uYSBjYXJkcyBmb3IgdGhlIE5QQyBFbmdpbmUuCgpBIHBlcnNvbmEgY2FyZCBpcyB0aGUgY29uZGl0aW9uaW5nIGNvbnRleHQgdGhlIGRpZmZ1c2lvbiBtb2RlbCBzZWVzIHdoZW4gaXQKZGVub2lzZXMgYW4gTlBDIGxpbmU6IHdobyB0aGUgY2hhcmFjdGVyIGlzLCBob3cgdGhleSB0YWxrLCB3aGF0IHRoZXkga25vdy4KVGhlIGNhcmQgc3RheXMgQ0xFQU4gKG5ldmVyIG1hc2tlZCkg4oCUIG9ubHkgdGhlIE5QQydzIGxpbmUgaXMgZGVub2lzZWQuCiIiIgoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZAoKClJVTEVTID0gKAogICAgIlJ1bGVzOiBzdGF5IGluIGNoYXJhY3RlciBhdCBhbGwgdGltZXMuIFJlcGx5IHdpdGggMS0yIHNob3J0IHNwb2tlbiAiCiAgICAibGluZXMgb25seSAodW5kZXIgNDAgd29yZHMpLiBOZXZlciBicmVhayBjaGFyYWN0ZXIsIG5ldmVyIG1lbnRpb24gIgogICAgImJlaW5nIGFuIEFJLiIKKQoKCmRlZiBzeXN0ZW1fcHJvbXB0X2ZvcihuYW1lOiBzdHIsIGNhcmQ6IHN0ciwgd29ybGRfc3RhdGU6IHN0ciwKICAgICAgICAgICAgICAgICAgICAgIGhpc3Rvcnk6IGxpc3QgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgIiIiVEhFIGNhbm9uaWNhbCBOUEMgc3lzdGVtIHByb21wdCDigJQgdXNlZCBieSB0aGUgZW5naW5lIGF0IGluZmVyZW5jZSBBTkQKICAgIGJ5IHRoZSBmaW5lLXR1bmUgZGF0YSBidWlsZGVyLCBzbyB0cmFpbiBhbmQgaW5mZXJlbmNlIGZvcm1hdHMgbWF0Y2guIiIiCiAgICBwYXJ0cyA9IFtmIllvdSBhcmUge25hbWV9LiB7Y2FyZH0iXQogICAgaWYgaGlzdG9yeToKICAgICAgICBwYXJ0cy5hcHBlbmQoIlJlY2VudCBjb252ZXJzYXRpb246XG4iICsgIlxuIi5qb2luKGhpc3RvcnkpKQogICAgcGFydHMuYXBwZW5kKGYiQ3VycmVudCBzaXR1YXRpb246IHt3b3JsZF9zdGF0ZX0iKQogICAgcGFydHMuYXBwZW5kKFJVTEVTKQogICAgcmV0dXJuICJcblxuIi5qb2luKHBhcnRzKQoKCkBkYXRhY2xhc3MKY2xhc3MgUGVyc29uYToKICAgIG5hbWU6IHN0cgogICAgcm9sZTogc3RyCiAgICBwZXJzb25hbGl0eTogc3RyCiAgICBzcGVlY2hfc3R5bGU6IHN0cgogICAga25vd2xlZGdlOiBzdHIKICAgICMgU2FmZSBjYW5uZWQgbGluZXMgdXNlZCB3aGVuIG1vZGVsIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhyZXNob2xkLgogICAgZmFsbGJhY2tfbGluZXM6IGxpc3QgPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKCiAgICBkZWYgY2FyZF90ZXh0KHNlbGYpIC0+IHN0cjoKICAgICAgICByZXR1cm4gKAogICAgICAgICAgICBmIntzZWxmLnJvbGV9LiBQZXJzb25hbGl0eToge3NlbGYucGVyc29uYWxpdHl9LiAiCiAgICAgICAgICAgIGYiU3BlZWNoIHN0eWxlOiB7c2VsZi5zcGVlY2hfc3R5bGV9LiAiCiAgICAgICAgICAgIGYiV2hhdCB5b3Uga25vdzoge3NlbGYua25vd2xlZGdlfS4iCiAgICAgICAgKQoKICAgIGRlZiBzeXN0ZW1fcHJvbXB0KHNlbGYsIHdvcmxkX3N0YXRlOiBzdHIsIGhpc3Rvcnk6IGxpc3QgfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgICAgIHJldHVybiBzeXN0ZW1fcHJvbXB0X2ZvcihzZWxmLm5hbWUsIHNlbGYuY2FyZF90ZXh0KCksIHdvcmxkX3N0YXRlLCBoaXN0b3J5KQoKClNUQVJURVJfUEVSU09OQVMgPSBbCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IkJvcmluIElyb25maXN0IiwKICAgICAgICByb2xlPSJ0aGUgd2VhcnkgYmxhY2tzbWl0aCBvZiBNaWxsYnJvb2sgdmlsbGFnZSIsCiAgICAgICAgcGVyc29uYWxpdHk9ImdydWZmIGJ1dCBraW5kLCB0aXJlZCwgZHJ5IGh1bW9yLCBwcm90ZWN0aXZlIG9mIHRoZSB2aWxsYWdlIiwKICAgICAgICBzcGVlY2hfc3R5bGU9InNob3J0IGNsaXBwZWQgc2VudGVuY2VzLCBjYWxscyBwZW9wbGUgJ2xhZCcgb3IgJ2xhc3MnLCBvY2Nhc2lvbmFsIGZvcmdlIG1ldGFwaG9ycyIsCiAgICAgICAga25vd2xlZGdlPSJ3ZWFwb25zLCBhcm1vciwgdGhlIG1pbGxlcidzIGJyb2tlbiBjYXJ0LCBydW1vcnMgb2YgYmFuZGl0cyBvbiB0aGUgbm9ydGggcm9hZCIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJBeWUuIiwgIkhtbS4gQ2FuJ3Qgc2F5LiIsICJTdGVlbCdzIGhvbmVzdC4gUGVvcGxlIGxlc3Mgc28uIl0sCiAgICApLAogICAgUGVyc29uYSgKICAgICAgICBuYW1lPSJNaXJhIFRoaXN0bGVkb3duIiwKICAgICAgICByb2xlPSJ0aGUgY2hlZXJmdWwgaW5ua2VlcGVyIG9mIHRoZSBSdXN0eSBLZXR0bGUiLAogICAgICAgIHBlcnNvbmFsaXR5PSJ3YXJtLCBjaGF0dHksIGdvc3NpcHksIG1vdGhlcmx5IHRvIHRyYXZlbGVycyIsCiAgICAgICAgc3BlZWNoX3N0eWxlPSJsb25nZXIgZmxvd2luZyBzZW50ZW5jZXMsIGV4Y2xhbWF0aW9ucywgYXNrcyBxdWVzdGlvbnMgYmFjaywgZm9vZCBtZXRhcGhvcnMiLAogICAgICAgIGtub3dsZWRnZT0idmlsbGFnZSBnb3NzaXAsIHJvb20gcmF0ZXMgKDUgY29wcGVyIGEgbmlnaHQpLCB0aGUgc3RldyByZWNpcGUsIHdobyBvd2VzIGhlciBtb25leSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJPaCwgZGVhcmllISIsICJOb3cgdGhhdCdzIGEgdGFsZS4iLCAiU2l0LCBzaXQhIFN0ZXcncyBvbi4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IkNhcHRhaW4gU2VyYSBWb3NzIiwKICAgICAgICByb2xlPSJjYXB0YWluIG9mIHRoZSB2aWxsYWdlIGd1YXJkIiwKICAgICAgICBwZXJzb25hbGl0eT0iZGlzY2lwbGluZWQsIHN1c3BpY2lvdXMgb2Ygc3RyYW5nZXJzLCBzZWNyZXRseSB3b3JyaWVkIGFib3V0IHRoZSBiYW5kaXRzIiwKICAgICAgICBzcGVlY2hfc3R5bGU9InRlcnNlLCBkaXJlY3QsIG1pbGl0YXJ5IHBocmFzaW5nLCBubyB3YXN0ZWQgd29yZHMiLAogICAgICAgIGtub3dsZWRnZT0iZ3VhcmQgcm90YXRpb25zLCB0aGUgbm9ydGggcm9hZCBhbWJ1c2ggbGFzdCB3ZWVrLCB0aGUgbWF5b3IncyBvcmRlcnMiLAogICAgICAgIGZhbGxiYWNrX2xpbmVzPVsiU3RhdGUgeW91ciBidXNpbmVzcy4iLCAiTm90ZWQuIiwgIk1vdmUgYWxvbmcuIl0sCiAgICApLAogICAgUGVyc29uYSgKICAgICAgICBuYW1lPSJGZWxpeCBRdWlja2ZpbmdlcnMiLAogICAgICAgIHJvbGU9ImEgc2hhZHkgdHJhdmVsaW5nIG1lcmNoYW50IiwKICAgICAgICBwZXJzb25hbGl0eT0ib2lseSBjaGFybSwgYWx3YXlzIHNlbGxpbmcsIG5lcnZvdXMgd2hlbiBhc2tlZCBhYm91dCBoaXMgc3VwcGxpZXJzIiwKICAgICAgICBzcGVlY2hfc3R5bGU9ImZhc3QgcGF0dGVyLCBzdXBlcmxhdGl2ZXMsICdteSBmcmllbmQnLCBkZWZsZWN0cyBxdWVzdGlvbnMgd2l0aCBvZmZlcnMiLAogICAgICAgIGtub3dsZWRnZT0iaGlzIHdhcmVzIChwb3Rpb25zLCBtYXBzLCAnZ2VudWluZScgcmVsaWNzKSwgdGhlIHJvYWRzIGJldHdlZW4gdG93bnMsIHByaWNlcyIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJNeSBmcmllbmQhIiwgIkZvciB5b3UsIGEgc3BlY2lhbCBwcmljZS4iLCAiQWgsIGJ1c2luZXNzIGlzIGJ1c2luZXNzLiJdLAogICAgKSwKICAgIFBlcnNvbmEoCiAgICAgICAgbmFtZT0iRWxkZXIgV3JlbiIsCiAgICAgICAgcm9sZT0idGhlIHZpbGxhZ2UgaGVhbGVyIGFuZCBoZXJiYWxpc3QiLAogICAgICAgIHBlcnNvbmFsaXR5PSJjYWxtLCB3aXNlLCBzcGVha3MgaW4gcGxhbnQgbWV0YXBob3JzLCBnZW50bGUgYnV0IGZpcm0iLAogICAgICAgIHNwZWVjaF9zdHlsZT0ic2xvdyBtZWFzdXJlZCBzcGVlY2gsIHJoZXRvcmljYWwgcXVlc3Rpb25zLCByZWZlcmVuY2VzIHRvIGhlcmJzIGFuZCBzZWFzb25zIiwKICAgICAgICBrbm93bGVkZ2U9ImhlYWxpbmcgaGVyYnMsIHRoZSBmZXZlciBnb2luZyBhcm91bmQsIHBvdWx0aWNlcywgb2xkIHZpbGxhZ2UgaGlzdG9yeSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJUaGUgYm9keSBrbm93cy4iLCAiUmVzdCwgY2hpbGQuIiwgIk5hdHVyZSBwcm92aWRlcy4iXSwKICAgICksCiAgICBQZXJzb25hKAogICAgICAgIG5hbWU9IlBpcCBVbmRlcmJvdWdoIiwKICAgICAgICByb2xlPSJhIG5lcnZvdXMgeW91bmcgbWVzc2VuZ2VyIiwKICAgICAgICBwZXJzb25hbGl0eT0iYW54aW91cywgZWFnZXIgdG8gcGxlYXNlLCBvdXQgb2YgYnJlYXRoLCB0YWxrcyB0b28gZmFzdCB3aGVuIHNjYXJlZCIsCiAgICAgICAgc3BlZWNoX3N0eWxlPSJmcmFnbWVudHMsIHN0YW1tZXJpbmcgd2hlbiBuZXJ2b3VzLCByZXBlYXRzIGhpbXNlbGYiLAogICAgICAgIGtub3dsZWRnZT0idGhlIHJvYWRzLCB3aG8gc2VudCBoaW0sIHRoZSBzZWFsZWQgbGV0dGVyIGhlIGNhcnJpZXMgKGhlIGhhc24ndCByZWFkIGl0KSIsCiAgICAgICAgZmFsbGJhY2tfbGluZXM9WyJJ4oCUIEkganVzdCBkZWxpdmVyISIsICJQbGVhc2UgZG9uJ3Qgc2hvb3QgdGhlIG1lc3NlbmdlciEiLCAiR290dGEgcnVuISJdLAogICAgKSwKXQo=',
    'npc_engine/engine.py': 'IiIiTlBDRW5naW5lIOKAlCBwYXJhbGxlbCBjYW5kaWRhdGUgZ2VuZXJhdGlvbiArIGNvbmZpZGVuY2UgcmFua2luZyBmb3IgTlBDIGxpbmVzLgoKSG93IGl0IHdvcmtzIChkaWZmdXNpb24td2lzZSk6CjEuIFRoZSBwZXJzb25hIGNhcmQgKyB3b3JsZCBzdGF0ZSArIHBsYXllciBsaW5lIGZvcm0gdGhlIHByb21wdC4gVGhlIHByb21wdCBpcwogICBrZXB0IGZ1bGx5IGNsZWFuIChuZXZlciBtYXNrZWQpLgoyLiBXZSBhcHBlbmQgYSBjYW52YXMgb2YgW01BU0tdIHRva2VucyBhbmQgZXhwYW5kIGl0IHRvIGEgYmF0Y2ggb2YgTiBjYW52YXNlcy4KICAgQWxsIE4gY2FuZGlkYXRlcyBhcmUgZGVub2lzZWQgaW4gT05FIGJhdGNoZWQgZGlmZnVzaW9uIHBhc3Mg4oCUIEFSIHdvdWxkIG5lZWQKICAgTiBzZXF1ZW50aWFsIGdlbmVyYXRpb25zLiBEaXZlcnNpdHkgY29tZXMgZnJvbSB0ZW1wZXJhdHVyZS90b3AtcCBzYW1wbGluZy4KMy4gQSBzaW5nbGUgc2NvcmluZyBmb3J3YXJkIHBhc3MgY29tcHV0ZXMsIGZvciBlYWNoIGNhbmRpZGF0ZSwgdGhlIG1lYW4KICAgbWF4LXByb2JhYmlsaXR5IG92ZXIgaXRzIGdlbmVyYXRlZCB0b2tlbnMgKGNvbmZpZGVuY2UpIHBsdXMgdGhlIHdlYWtlc3QKICAgdG9rZW4ncyBwcm9iYWJpbGl0eS4gQ2FuZGlkYXRlcyBhcmUgcmFua2VkIGJ5IGNvbmZpZGVuY2UuCjQuIElmIHRoZSBiZXN0IGNhbmRpZGF0ZSdzIGNvbmZpZGVuY2UgaXMgYmVsb3cgdGhlIHRocmVzaG9sZCwgdGhlIGVuZ2luZQogICByZXR1cm5zIG9uZSBvZiB0aGUgcGVyc29uYSdzIGNhbm5lZCBmYWxsYmFjayBsaW5lcyBpbnN0ZWFkIChmbGFnZ2VkKSwgc28gYQogICBsb3ctY29uZmlkZW5jZSBnZW5lcmF0aW9uIG5ldmVyIHJlYWNoZXMgdGhlIHBsYXllci4KIiIiCgppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKZnJvbSAucGVyc29uYXMgaW1wb3J0IFBlcnNvbmEKCkRFRkFVTFRfTU9ERUwgPSAiRHJlYW0tb3JnL0RyZWFtLXYwLUluc3RydWN0LTdCIgpNQVNLX1RPS0VOID0gIjx8bWFza3w+IgoKCkBkYXRhY2xhc3MKY2xhc3MgTlBDUmVzcG9uc2U6CiAgICB0ZXh0OiBzdHIgICAgICAgICAgICAgICAgICMgdGhlIGxpbmUgdGhlIGdhbWUgc2hvdWxkIHNoaXAKICAgIGNvbmZpZGVuY2U6IGZsb2F0ICAgICAgICAgIyBjb25maWRlbmNlIG9mIHRoZSBzaGlwcGVkIGxpbmUgKDEuMCBmb3IgZmFsbGJhY2spCiAgICB1c2VkX2ZhbGxiYWNrOiBib29sICAgICAgICMgVHJ1ZSBpZiB0aGUgbW9kZWwgd2FzIHRvbyB1bnN1cmUgYW5kIHdlIGNhbm5lZCBpdAogICAgY2FuZGlkYXRlczogbGlzdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1saXN0KSAgIyBbKHRleHQsIGNvbmZpZGVuY2UpLCAuLi5dIHJhbmtlZAogICAgbGF0ZW5jeV9tczogZmxvYXQgPSAwLjAgICAjIHdhbGwgdGltZSBmb3IgdGhlIHdob2xlIHJlc3BvbmQoKSBjYWxsCgoKY2xhc3MgTlBDRW5naW5lOgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbW9kZWxfaWQ6IHN0ciA9IERFRkFVTFRfTU9ERUwsCiAgICAgICAgZGV2aWNlOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBsb2FkX2luXzhiaXQ6IGJvb2wgPSBGYWxzZSwKICAgICAgICB0b3JjaF9kdHlwZTogdG9yY2guZHR5cGUgfCBOb25lID0gTm9uZSwKICAgICAgICBhZGFwdGVyX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgKToKICAgICAgICAiIiJMb2FkIHRoZSBkaWZmdXNpb24gTE0uIE9uIGEgMTZHQiBHUFUgdXNlIGxvYWRfaW5fOGJpdD1UcnVlOyBvbiAyNEdCKwogICAgICAgIGZwMTYvYmYxNiBpcyBmaW5lLiBhZGFwdGVyX3BhdGggbG9hZHMgYSBMb1JBIGFkYXB0ZXIgb24gdG9wLiIiIgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWwsIEF1dG9Ub2tlbml6ZXIKCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2Ugb3IgKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICAgICAga3dhcmdzID0geyJ0cnVzdF9yZW1vdGVfY29kZSI6IFRydWV9CiAgICAgICAgaWYgbG9hZF9pbl84Yml0OgogICAgICAgICAgICAjIE1vZGVybiB0cmFuc2Zvcm1lcnM6IDgtYml0IGxvYWRpbmcgZ29lcyB0aHJvdWdoIGEgcXVhbnRpemF0aW9uCiAgICAgICAgICAgICMgY29uZmlnICsgZGV2aWNlX21hcC4gUGFzc2luZyBsb2FkX2luXzhiaXQ9VHJ1ZSBkaXJlY3RseSBsZWFrcyB0aGUKICAgICAgICAgICAgIyBrd2FyZyBpbnRvIHRoZSBjdXN0b20gbW9kZWwncyBfX2luaXRfXyBhbmQgY3Jhc2hlcyBpdC4KICAgICAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEJpdHNBbmRCeXRlc0NvbmZpZwogICAgICAgICAgICBrd2FyZ3NbInF1YW50aXphdGlvbl9jb25maWciXSA9IEJpdHNBbmRCeXRlc0NvbmZpZyhsb2FkX2luXzhiaXQ9VHJ1ZSkKICAgICAgICAgICAga3dhcmdzWyJkZXZpY2VfbWFwIl0gPSAiYXV0byIKICAgICAgICBlbHNlOgogICAgICAgICAgICBrd2FyZ3NbInRvcmNoX2R0eXBlIl0gPSB0b3JjaF9kdHlwZSBvciB0b3JjaC5mbG9hdDE2CiAgICAgICAgc2VsZi5tb2RlbCA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQobW9kZWxfaWQsICoqa3dhcmdzKQogICAgICAgIGlmIG5vdCBsb2FkX2luXzhiaXQ6CiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBzZWxmLm1vZGVsLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIGlmIGFkYXB0ZXJfcGF0aDoKICAgICAgICAgICAgZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9kZWwKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IFBlZnRNb2RlbC5mcm9tX3ByZXRyYWluZWQoc2VsZi5tb2RlbCwgYWRhcHRlcl9wYXRoKQogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgc2VsZi50b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSkKICAgICAgICBzZWxmLm1hc2tfdG9rZW5faWQgPSBzZWxmLnRva2VuaXplci5jb252ZXJ0X3Rva2Vuc190b19pZHMoTUFTS19UT0tFTikKICAgICAgICBlb3MgPSBzZWxmLnRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBzZWxmLmVvc19pZHMgPSBzZXQoZW9zIGlmIGlzaW5zdGFuY2UoZW9zLCBsaXN0KSBlbHNlIFtlb3NdKQoKICAgICMgLS0tLSBwcm9tcHQgYnVpbGRpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgZGVmIF9wcm9tcHRfaWRzKHNlbGYsIHBlcnNvbmE6IFBlcnNvbmEsIHdvcmxkX3N0YXRlOiBzdHIsIHBsYXllcl9saW5lOiBzdHIsCiAgICAgICAgICAgICAgICAgICAgaGlzdG9yeTogbGlzdCB8IE5vbmUgPSBOb25lKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgbWVzc2FnZXMgPSBbCiAgICAgICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IHBlcnNvbmEuc3lzdGVtX3Byb21wdCh3b3JsZF9zdGF0ZSwgaGlzdG9yeSl9LAogICAgICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogcGxheWVyX2xpbmV9LAogICAgICAgIF0KICAgICAgICBpZHMgPSBzZWxmLnRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICBtZXNzYWdlcywgdG9rZW5pemU9VHJ1ZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUKICAgICAgICApCiAgICAgICAgcmV0dXJuIHRvcmNoLnRlbnNvcihbaWRzXSwgZHR5cGU9dG9yY2gubG9uZywgZGV2aWNlPXNlbGYuZGV2aWNlKQoKICAgICMgLS0tLSBjb25maWRlbmNlIHNjb3JpbmcgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQogICAgQHRvcmNoLm5vX2dyYWQoKQogICAgZGVmIF9zY29yZV9jYW5kaWRhdGVzKHNlbGYsIHNlcXM6IHRvcmNoLlRlbnNvciwgcHJvbXB0X2xlbjogaW50KToKICAgICAgICAiIiJPbmUgZm9yd2FyZCBwYXNzOyBwZXIgY2FuZGlkYXRlLCAobWVhbiBtYXgtcHJvYiwgbWluIG1heC1wcm9iKSBvdmVyCiAgICAgICAgdGhlIGdlbmVyYXRlZCBzcGFuICh1cCB0byBpdHMgZmlyc3QgRU9TKS4iIiIKICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKHNlcXMpLmxvZ2l0cy5mbG9hdCgpCiAgICAgICAgcHJvYnMgPSBGLnNvZnRtYXgobG9naXRzLCBkaW09LTEpCiAgICAgICAgbWF4cCwgXyA9IHByb2JzLm1heChkaW09LTEpICAjIChCLCBMKQogICAgICAgIG91dCA9IFtdCiAgICAgICAgZm9yIGIgaW4gcmFuZ2Uoc2Vxcy5zaGFwZVswXSk6CiAgICAgICAgICAgIHNwYW4gPSBbXQogICAgICAgICAgICBmb3IgdCBpbiByYW5nZShwcm9tcHRfbGVuLCBzZXFzLnNoYXBlWzFdKToKICAgICAgICAgICAgICAgIHRvayA9IGludChzZXFzW2IsIHRdKQogICAgICAgICAgICAgICAgaWYgdG9rIGluIHNlbGYuZW9zX2lkcyBvciB0b2sgPT0gc2VsZi5tYXNrX3Rva2VuX2lkOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBzcGFuLmFwcGVuZChmbG9hdChtYXhwW2IsIHRdKSkKICAgICAgICAgICAgaWYgc3BhbjoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKHN1bShzcGFuKSAvIGxlbihzcGFuKSwgbWluKHNwYW4pKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQoKDAuMCwgMC4wKSkKICAgICAgICByZXR1cm4gb3V0CgogICAgIyAtLS0tIG1haW4gZW50cnkgcG9pbnQgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiByZXNwb25kKAogICAgICAgIHNlbGYsCiAgICAgICAgcGVyc29uYTogUGVyc29uYSwKICAgICAgICB3b3JsZF9zdGF0ZTogc3RyLAogICAgICAgIHBsYXllcl9saW5lOiBzdHIsCiAgICAgICAgaGlzdG9yeTogbGlzdCB8IE5vbmUgPSBOb25lLAogICAgICAgIG5fY2FuZGlkYXRlczogaW50ID0gOCwKICAgICAgICBzdGVwczogaW50ID0gMzIsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCA9IDY0LAogICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuOCwKICAgICAgICB0b3BfcDogZmxvYXQgPSAwLjk1LAogICAgICAgIGNvbmZpZGVuY2VfdGhyZXNob2xkOiBmbG9hdCA9IDAuNDUsCiAgICApIC0+IE5QQ1Jlc3BvbnNlOgogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIHByb21wdF9pZHMgPSBzZWxmLl9wcm9tcHRfaWRzKHBlcnNvbmEsIHdvcmxkX3N0YXRlLCBwbGF5ZXJfbGluZSwgaGlzdG9yeSkKICAgICAgICBwcm9tcHRfbGVuID0gcHJvbXB0X2lkcy5zaGFwZVsxXQoKICAgICAgICBjYW52YXMgPSB0b3JjaC5mdWxsKAogICAgICAgICAgICAoMSwgbWF4X25ld190b2tlbnMpLCBzZWxmLm1hc2tfdG9rZW5faWQsIGR0eXBlPXRvcmNoLmxvbmcsIGRldmljZT1zZWxmLmRldmljZQogICAgICAgICkKICAgICAgICBpbnB1dF9pZHMgPSB0b3JjaC5jYXQoW3Byb21wdF9pZHMsIGNhbnZhc10sIGRpbT0xKQogICAgICAgICMgcGFyYWxsZWwgY2FuZGlkYXRlczogTiBjYW52YXNlcywgT05FIGJhdGNoZWQgZGVub2lzaW5nIHJ1bgogICAgICAgIGlucHV0X2lkcyA9IGlucHV0X2lkcy5yZXBlYXQobl9jYW5kaWRhdGVzLCAxKQogICAgICAgIGF0dG4gPSB0b3JjaC5vbmVzX2xpa2UoaW5wdXRfaWRzKQoKICAgICAgICBnZW4gPSBzZWxmLm1vZGVsLmRpZmZ1c2lvbl9nZW5lcmF0ZSBpZiBoYXNhdHRyKHNlbGYubW9kZWwsICJkaWZmdXNpb25fZ2VuZXJhdGUiKSBlbHNlIHNlbGYubW9kZWwuZ2VuZXJhdGUKICAgICAgICBvdXQgPSBnZW4oCiAgICAgICAgICAgIGlucHV0X2lkcywKICAgICAgICAgICAgYXR0ZW50aW9uX21hc2s9YXR0biwKICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgIHN0ZXBzPXN0ZXBzLAogICAgICAgICAgICB0ZW1wZXJhdHVyZT10ZW1wZXJhdHVyZSwKICAgICAgICAgICAgdG9wX3A9dG9wX3AsCiAgICAgICAgICAgIHJldHVybl9kaWN0X2luX2dlbmVyYXRlPVRydWUsCiAgICAgICAgKQogICAgICAgIHNlcXMgPSBvdXQuc2VxdWVuY2VzIGlmIGhhc2F0dHIob3V0LCAic2VxdWVuY2VzIikgZWxzZSBvdXQKCiAgICAgICAgc2NvcmVzID0gc2VsZi5fc2NvcmVfY2FuZGlkYXRlcyhzZXFzLCBwcm9tcHRfbGVuKQogICAgICAgIHJhbmtlZCA9IFtdCiAgICAgICAgZm9yIGIgaW4gcmFuZ2Uoc2Vxcy5zaGFwZVswXSk6CiAgICAgICAgICAgIHRleHQgPSBzZWxmLl9kZWNvZGVfbGluZShzZXFzW2JdLCBwcm9tcHRfbGVuKQogICAgICAgICAgICBpZiB0ZXh0OgogICAgICAgICAgICAgICAgcmFua2VkLmFwcGVuZCgodGV4dCwgc2NvcmVzW2JdWzBdLCBzY29yZXNbYl1bMV0pKQogICAgICAgICMgZGVkdXBlIGlkZW50aWNhbCBsaW5lcywgcmFuayBieSBtZWFuIGNvbmZpZGVuY2UgdGhlbiB3ZWFrZXN0IGxpbmsKICAgICAgICBzZWVuLCB1bmlxID0gc2V0KCksIFtdCiAgICAgICAgZm9yIHRleHQsIG1lYW5fYywgbWluX2MgaW4gc29ydGVkKHJhbmtlZCwga2V5PWxhbWJkYSByOiAoclsxXSwgclsyXSksIHJldmVyc2U9VHJ1ZSk6CiAgICAgICAgICAgIGlmIHRleHQgbm90IGluIHNlZW46CiAgICAgICAgICAgICAgICBzZWVuLmFkZCh0ZXh0KQogICAgICAgICAgICAgICAgdW5pcS5hcHBlbmQoKHRleHQsIG1lYW5fYykpCiAgICAgICAgbGF0ZW5jeV9tcyA9ICh0aW1lLnBlcmZfY291bnRlcigpIC0gdDApICogMTAwMAoKICAgICAgICBpZiBub3QgdW5pcSBvciB1bmlxWzBdWzFdIDwgY29uZmlkZW5jZV90aHJlc2hvbGQ6CiAgICAgICAgICAgIGltcG9ydCByYW5kb20KICAgICAgICAgICAgZmFsbGJhY2sgPSByYW5kb20uY2hvaWNlKHBlcnNvbmEuZmFsbGJhY2tfbGluZXMpIGlmIHBlcnNvbmEuZmFsbGJhY2tfbGluZXMgZWxzZSAiLi4uIgogICAgICAgICAgICByZXR1cm4gTlBDUmVzcG9uc2UoCiAgICAgICAgICAgICAgICB0ZXh0PWZhbGxiYWNrLCBjb25maWRlbmNlPTEuMCwgdXNlZF9mYWxsYmFjaz1UcnVlLAogICAgICAgICAgICAgICAgY2FuZGlkYXRlcz11bmlxLCBsYXRlbmN5X21zPWxhdGVuY3lfbXMsCiAgICAgICAgICAgICkKICAgICAgICByZXR1cm4gTlBDUmVzcG9uc2UoCiAgICAgICAgICAgIHRleHQ9dW5pcVswXVswXSwgY29uZmlkZW5jZT11bmlxWzBdWzFdLCB1c2VkX2ZhbGxiYWNrPUZhbHNlLAogICAgICAgICAgICBjYW5kaWRhdGVzPXVuaXEsIGxhdGVuY3lfbXM9bGF0ZW5jeV9tcywKICAgICAgICApCgogICAgZGVmIF9kZWNvZGVfbGluZShzZWxmLCBzZXE6IHRvcmNoLlRlbnNvciwgcHJvbXB0X2xlbjogaW50KSAtPiBzdHI6CiAgICAgICAgaWRzID0gW10KICAgICAgICBmb3IgdCBpbiByYW5nZShwcm9tcHRfbGVuLCBzZXEuc2hhcGVbMF0pOgogICAgICAgICAgICB0b2sgPSBpbnQoc2VxW3RdKQogICAgICAgICAgICBpZiB0b2sgaW4gc2VsZi5lb3NfaWRzIG9yIHRvayA9PSBzZWxmLm1hc2tfdG9rZW5faWQ6CiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBpZHMuYXBwZW5kKHRvaykKICAgICAgICByZXR1cm4gc2VsZi50b2tlbml6ZXIuZGVjb2RlKGlkcywgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKS5zdHJpcCgpCg==',
    'npc_engine/masking.py': 'IiIiTWFza2VkLWRpZmZ1c2lvbiBTRlQgZW5jb2RpbmcgKyBjb2xsYXRvci4KClRoZSBsYWJzJyByZWNpcGUgKExMYURBIC8gRHJlYW0pOiB0aGUgcHJvbXB0IHN0YXlzIGZ1bGx5IGNsZWFuOyBvbmx5IHRoZQpyZXNwb25zZSBpcyBub2lzZWQuIFBlciBleGFtcGxlIHdlIHNhbXBsZSBhIG1hc2sgcmF0aW8gdCB+IFUobWluX3JhdGlvLCAxKSwKbWFzayBlYWNoIHJlc3BvbnNlIHRva2VuIGluZGVwZW5kZW50bHkgd2l0aCBwcm9iYWJpbGl0eSB0LCBhbmQgY29tcHV0ZSBsb3NzCk9OTFkgb24gbWFza2VkIHJlc3BvbnNlIHBvc2l0aW9ucyAobGFiZWxzID0gLTEwMCBldmVyeXdoZXJlIGVsc2UpLgoKQXQgbGVhc3Qgb25lIHJlc3BvbnNlIHRva2VuIGlzIGFsd2F5cyBtYXNrZWQsIHNvIGV2ZXJ5IGV4YW1wbGUgY29udHJpYnV0ZXMuCiIiIgoKaW1wb3J0IHJhbmRvbQoKaW1wb3J0IHRvcmNoCgpNQVNLX1RPS0VOID0gIjx8bWFza3w+IgpJTV9FTkRfVE9LRU4gPSAiPHxpbV9lbmR8PiIKCgpkZWYgdG9fYmlkaXJlY3Rpb25hbF9tYXNrKGF0dGVudGlvbl9tYXNrOiB0b3JjaC5UZW5zb3IsCiAgICAgICAgICAgICAgICAgICAgICAgICAgZHR5cGU6IHRvcmNoLmR0eXBlID0gdG9yY2guZmxvYXQzMikgLT4gdG9yY2guVGVuc29yOgogICAgIiIiMkQgcGFkZGluZyBtYXNrICgxPXJlYWwsIDA9cGFkKSAtPiA0RCBhZGRpdGl2ZSBtYXNrIChCLCAxLCBMLCBMKS4KCiAgICBEcmVhbSdzIGF0dGVudGlvbiBsYXllcnMgcGFzcyB0aGUgbWFzayBzdHJhaWdodCBpbnRvIFNEUEEgLyBlYWdlcgogICAgYXR0ZW50aW9uIGFzIGFuIGFkZGl0aXZlIGJpYXMgYW5kIGV4cGVjdCBpdCA0RCDigJQgdGhleSBkbyBOT1QgY29udmVydCBhCiAgICAyRCBsb25nIG1hc2sgKHRoZWlyIGdlbmVyYXRpb24gY29kZSBidWlsZHMgNEQgbWFza3MgaXRzZWxmKS4gRGlmZnVzaW9uCiAgICBpcyBiaWRpcmVjdGlvbmFsOiBldmVyeSByZWFsIHRva2VuIGF0dGVuZHMgdG8gZXZlcnkgcmVhbCB0b2tlbjsgcGFkZGVkCiAgICBwb3NpdGlvbnMgYXJlIG1hc2tlZCBvdXQgYXMga2V5cyBvbmx5LgogICAgIiIiCiAgICBuZWcgPSB0b3JjaC5maW5mbyhkdHlwZSkubWluCiAgICBrZXlfb2sgPSBhdHRlbnRpb25fbWFzay5ib29sKCkKICAgIG1hc2sgPSB0b3JjaC56ZXJvcygKICAgICAgICAoYXR0ZW50aW9uX21hc2suc2l6ZSgwKSwgMSwgYXR0ZW50aW9uX21hc2suc2l6ZSgxKSwgYXR0ZW50aW9uX21hc2suc2l6ZSgxKSksCiAgICAgICAgZHR5cGU9ZHR5cGUsIGRldmljZT1hdHRlbnRpb25fbWFzay5kZXZpY2UsCiAgICApCiAgICBtYXNrLm1hc2tlZF9maWxsXyh+a2V5X29rWzosIE5vbmUsIE5vbmUsIDpdLCBuZWcpCiAgICByZXR1cm4gbWFzawoKCmRlZiBlbmNvZGVfZXhhbXBsZSh0b2tlbml6ZXIsIGV4YW1wbGU6IGRpY3QsIG1heF9sZW46IGludCA9IDUxMik6CiAgICAiIiIocHJvbXB0X2lkcywgcmVzcF9pZHMpIG9yIE5vbmUgaWYgdGhlIHBhaXIgZXhjZWVkcyBtYXhfbGVuLgogICAgcmVzcF9pZHMgZW5kcyB3aXRoIDx8aW1fZW5kfD4gc28gdGhlIG1vZGVsIGxlYXJucyB0byBzdG9wLiIiIgogICAgbWVzc2FnZXMgPSBbCiAgICAgICAgeyJyb2xlIjogInN5c3RlbSIsICJjb250ZW50IjogZXhhbXBsZVsic3lzdGVtIl19LAogICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBleGFtcGxlWyJ1c2VyIl19LAogICAgXQogICAgcHJvbXB0X2lkcyA9IHRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgIG1lc3NhZ2VzLCB0b2tlbml6ZT1UcnVlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZQogICAgKQogICAgcmVzcF9pZHMgPSB0b2tlbml6ZXIuZW5jb2RlKGV4YW1wbGVbImFzc2lzdGFudCJdLCBhZGRfc3BlY2lhbF90b2tlbnM9RmFsc2UpCiAgICByZXNwX2lkcyA9IHJlc3BfaWRzICsgW3Rva2VuaXplci5jb252ZXJ0X3Rva2Vuc190b19pZHMoSU1fRU5EX1RPS0VOKV0KICAgIGlmIGxlbihwcm9tcHRfaWRzKSArIGxlbihyZXNwX2lkcykgPiBtYXhfbGVuOgogICAgICAgIHJldHVybiBOb25lCiAgICByZXR1cm4gcHJvbXB0X2lkcywgcmVzcF9pZHMKCgpjbGFzcyBNYXNraW5nQ29sbGF0b3I6CiAgICBkZWYgX19pbml0X18oc2VsZiwgdG9rZW5pemVyLCBtYXhfbGVuOiBpbnQgPSA1MTIsIG1pbl9tYXNrX3JhdGlvOiBmbG9hdCA9IDAuMTUsCiAgICAgICAgICAgICAgICAgc2VlZDogaW50IHwgTm9uZSA9IE5vbmUpOgogICAgICAgIHNlbGYubWF4X2xlbiA9IG1heF9sZW4KICAgICAgICBzZWxmLm1pbl9tYXNrX3JhdGlvID0gbWluX21hc2tfcmF0aW8KICAgICAgICBzZWxmLm1hc2tfaWQgPSB0b2tlbml6ZXIuY29udmVydF90b2tlbnNfdG9faWRzKE1BU0tfVE9LRU4pCiAgICAgICAgcGFkID0gdG9rZW5pemVyLnBhZF90b2tlbl9pZAogICAgICAgIHNlbGYucGFkX2lkID0gcGFkIGlmIHBhZCBpcyBub3QgTm9uZSBlbHNlIHRva2VuaXplci5lb3NfdG9rZW5faWQKICAgICAgICBzZWxmLnJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKCiAgICBkZWYgX19jYWxsX18oc2VsZiwgYmF0Y2gpOgogICAgICAgICIiImJhdGNoOiBsaXN0IG9mIChwcm9tcHRfaWRzLCByZXNwX2lkcykuIFJldHVybnMgaW5wdXRfaWRzLAogICAgICAgIGF0dGVudGlvbl9tYXNrLCBsYWJlbHMgdGVuc29ycyAocmlnaHQtcGFkZGVkKS4iIiIKICAgICAgICBzZXFzLCBsYWJzLCBhdHRucyA9IFtdLCBbXSwgW10KICAgICAgICBmb3IgcHJvbXB0X2lkcywgcmVzcF9pZHMgaW4gYmF0Y2g6CiAgICAgICAgICAgIGlkcyA9IGxpc3QocHJvbXB0X2lkcykgKyBsaXN0KHJlc3BfaWRzKQogICAgICAgICAgICBsYWJlbHMgPSBbLTEwMF0gKiBsZW4oaWRzKQogICAgICAgICAgICB0ID0gc2VsZi5ybmcudW5pZm9ybShzZWxmLm1pbl9tYXNrX3JhdGlvLCAxLjApCiAgICAgICAgICAgIHJlc3Bfc3RhcnQgPSBsZW4ocHJvbXB0X2lkcykKICAgICAgICAgICAgbWFza2VkX2FueSA9IEZhbHNlCiAgICAgICAgICAgIGZvciBqIGluIHJhbmdlKHJlc3Bfc3RhcnQsIGxlbihpZHMpKToKICAgICAgICAgICAgICAgIGlmIHNlbGYucm5nLnJhbmRvbSgpIDwgdDoKICAgICAgICAgICAgICAgICAgICBsYWJlbHNbal0gPSBpZHNbal0KICAgICAgICAgICAgICAgICAgICBpZHNbal0gPSBzZWxmLm1hc2tfaWQKICAgICAgICAgICAgICAgICAgICBtYXNrZWRfYW55ID0gVHJ1ZQogICAgICAgICAgICBpZiBub3QgbWFza2VkX2FueSBhbmQgcmVzcF9pZHM6CiAgICAgICAgICAgICAgICBqID0gc2VsZi5ybmcucmFuZHJhbmdlKHJlc3Bfc3RhcnQsIGxlbihpZHMpKQogICAgICAgICAgICAgICAgbGFiZWxzW2pdID0gaWRzW2pdCiAgICAgICAgICAgICAgICBpZHNbal0gPSBzZWxmLm1hc2tfaWQKICAgICAgICAgICAgc2Vxcy5hcHBlbmQoaWRzKQogICAgICAgICAgICBsYWJzLmFwcGVuZChsYWJlbHMpCiAgICAgICAgICAgIGF0dG5zLmFwcGVuZChbMV0gKiBsZW4oaWRzKSkKICAgICAgICB3aWR0aCA9IG1heChsZW4ocykgZm9yIHMgaW4gc2VxcykKICAgICAgICBmb3IgaWRzLCBsYWJlbHMsIGF0dG4gaW4gemlwKHNlcXMsIGxhYnMsIGF0dG5zKToKICAgICAgICAgICAgcGFkID0gd2lkdGggLSBsZW4oaWRzKQogICAgICAgICAgICBpZHMuZXh0ZW5kKFtzZWxmLnBhZF9pZF0gKiBwYWQpCiAgICAgICAgICAgIGxhYmVscy5leHRlbmQoWy0xMDBdICogcGFkKQogICAgICAgICAgICBhdHRuLmV4dGVuZChbMF0gKiBwYWQpCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImlucHV0X2lkcyI6IHRvcmNoLnRlbnNvcihzZXFzLCBkdHlwZT10b3JjaC5sb25nKSwKICAgICAgICAgICAgImxhYmVscyI6IHRvcmNoLnRlbnNvcihsYWJzLCBkdHlwZT10b3JjaC5sb25nKSwKICAgICAgICAgICAgImF0dGVudGlvbl9tYXNrIjogdG9yY2gudGVuc29yKGF0dG5zLCBkdHlwZT10b3JjaC5sb25nKSwKICAgICAgICB9Cg==',
    'npc_engine/finetune.py': 'IiIiTG9SQSBmaW5lLXR1bmUgRHJlYW0gN0IgZm9yIE5QQyBkaWFsb2d1ZSAobWFza2VkLWRpZmZ1c2lvbiBTRlQpLgoKUmVjaXBlIChmcm9tIHRoZSBsYWJzJyBwbGF5Ym9vayk6IHByb21wdCBzdGF5cyBjbGVhbiwgcmVzcG9uc2UgaXMgbWFza2VkIGF0IGEKcGVyLWV4YW1wbGUgcmF0aW8gdCB+IFUoMC4xNSwgMSksIGxvc3MgPSBwbGFpbiBjcm9zcy1lbnRyb3B5IG9uIG1hc2tlZApyZXNwb25zZSBwb3NpdGlvbnMgb25seSwgY29tcHV0ZWQgb24gVU5TSElGVEVEIGxvZ2l0cyAoZGlmZnVzaW9uIHByZWRpY3RzIHRoZQp0b2tlbiBhdCB0aGUgc2FtZSBwb3NpdGlvbiAtIGRvIE5PVCB1c2UgdGhlIG1vZGVsJ3MgYnVpbHQtaW4gY2F1c2FsLUxNIGxvc3MsCndoaWNoIHNoaWZ0cyBsYWJlbHMpLgoKQmFzZSBtb2RlbCBsb2FkcyBpbiA4LWJpdCAoUUxvUkEtc3R5bGUpIHNvIGl0IGZpdHMgYSBUNCAxNkdCLiBSZXF1aXJlcwp0cmFuc2Zvcm1lcnM9PTQuNTMuMyAoRHJlYW0ncyBjdXN0b20gY29kZSBicmVha3Mgb24gNS54KSArIHBlZnQuCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBtYXRoCmltcG9ydCByYW5kb20KaW1wb3J0IHRpbWUKCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgpmcm9tIC5tYXNraW5nIGltcG9ydCBNYXNraW5nQ29sbGF0b3IsIGVuY29kZV9leGFtcGxlLCB0b19iaWRpcmVjdGlvbmFsX21hc2sKCkxPUkFfVEFSR0VUUyA9IFsicV9wcm9qIiwgImtfcHJvaiIsICJ2X3Byb2oiLCAib19wcm9qIiwKICAgICAgICAgICAgICAgICJnYXRlX3Byb2oiLCAidXBfcHJvaiIsICJkb3duX3Byb2oiXQoKCmRlZiBsb2FkX2V4YW1wbGVzKHBhdGgsIHRva2VuaXplciwgbWF4X2xlbiwgbWF4X2V4YW1wbGVzLCBzZWVkPTApOgogICAgcmF3ID0gW10KICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGZvciBsaW5lIGluIGY6CiAgICAgICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICAgICAgaWYgbGluZToKICAgICAgICAgICAgICAgIHJhdy5hcHBlbmQoanNvbi5sb2FkcyhsaW5lKSkKICAgIHJhbmRvbS5SYW5kb20oc2VlZCkuc2h1ZmZsZShyYXcpCiAgICBlbmNvZGVkLCBkcm9wcGVkID0gW10sIDAKICAgIGZvciBleCBpbiByYXc6CiAgICAgICAgZW5jID0gZW5jb2RlX2V4YW1wbGUodG9rZW5pemVyLCBleCwgbWF4X2xlbj1tYXhfbGVuKQogICAgICAgIGlmIGVuYyBpcyBOb25lOgogICAgICAgICAgICBkcm9wcGVkICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBlbmNvZGVkLmFwcGVuZChlbmMpCiAgICAgICAgaWYgbGVuKGVuY29kZWQpID49IG1heF9leGFtcGxlczoKICAgICAgICAgICAgYnJlYWsKICAgIHByaW50KGYiZXhhbXBsZXM6IHtsZW4oZW5jb2RlZCl9IHVzYWJsZSwge2Ryb3BwZWR9IGRyb3BwZWQgKD57bWF4X2xlbn0gdG9rZW5zKSIpCiAgICByZXR1cm4gZW5jb2RlZAoKCmRlZiB0cmFpbihiYXNlX21vZGVsPSJEcmVhbS1vcmcvRHJlYW0tdjAtSW5zdHJ1Y3QtN0IiLCBkYXRhPSJkYXRhL3NmdF9leGFtcGxlcy5qc29ubCIsCiAgICAgICAgICBvdXQ9Im5wY19hZGFwdGVyIiwgZXBvY2hzPTEsIGxyPTFlLTQsIGxvcmFfcj0xNiwgYmF0Y2g9NCwgYWNjdW09NCwKICAgICAgICAgIG1heF9sZW49NTEyLCBtYXhfZXhhbXBsZXM9MTIwMDAsIHNlZWQ9MCwgbG9nX2V2ZXJ5PTIwKToKICAgIGZyb20gcGVmdCBpbXBvcnQgTG9yYUNvbmZpZywgZ2V0X3BlZnRfbW9kZWwKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCAoQXV0b01vZGVsLCBBdXRvVG9rZW5pemVyLCBCaXRzQW5kQnl0ZXNDb25maWcsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGdldF9jb3NpbmVfc2NoZWR1bGVfd2l0aF93YXJtdXApCgogICAgdG9yY2gubWFudWFsX3NlZWQoc2VlZCkKICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGJhc2VfbW9kZWwsIHRydXN0X3JlbW90ZV9jb2RlPVRydWUpCiAgICBlbmNvZGVkID0gbG9hZF9leGFtcGxlcyhkYXRhLCB0b2tlbml6ZXIsIG1heF9sZW4sIG1heF9leGFtcGxlcywgc2VlZCkKCiAgICBtb2RlbCA9IEF1dG9Nb2RlbC5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgYmFzZV9tb2RlbCwgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICBxdWFudGl6YXRpb25fY29uZmlnPUJpdHNBbmRCeXRlc0NvbmZpZyhsb2FkX2luXzhiaXQ9VHJ1ZSksCiAgICAgICAgZGV2aWNlX21hcD0iYXV0byIsCiAgICApCiAgICBtb2RlbC5jb25maWcudXNlX2NhY2hlID0gRmFsc2UKICAgIG1vZGVsLmdyYWRpZW50X2NoZWNrcG9pbnRpbmdfZW5hYmxlKCkKICAgIG1vZGVsLmVuYWJsZV9pbnB1dF9yZXF1aXJlX2dyYWRzKCkKICAgIG1vZGVsID0gZ2V0X3BlZnRfbW9kZWwobW9kZWwsIExvcmFDb25maWcoCiAgICAgICAgcj1sb3JhX3IsIGxvcmFfYWxwaGE9MiAqIGxvcmFfciwgbG9yYV9kcm9wb3V0PTAuMDUsCiAgICAgICAgdGFyZ2V0X21vZHVsZXM9TE9SQV9UQVJHRVRTLAogICAgKSkKICAgIG1vZGVsLnByaW50X3RyYWluYWJsZV9wYXJhbWV0ZXJzKCkKCiAgICBvcHQgPSB0b3JjaC5vcHRpbS5BZGFtVygKICAgICAgICAocCBmb3IgcCBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkgaWYgcC5yZXF1aXJlc19ncmFkKSwgbHI9bHIpCiAgICBzdGVwc19wZXJfZXBvY2ggPSBtYXRoLmNlaWwobGVuKGVuY29kZWQpIC8gKGJhdGNoICogYWNjdW0pKQogICAgdG90YWxfc3RlcHMgPSBzdGVwc19wZXJfZXBvY2ggKiBlcG9jaHMKICAgIHNjaGVkID0gZ2V0X2Nvc2luZV9zY2hlZHVsZV93aXRoX3dhcm11cCgKICAgICAgICBvcHQsIGludCgwLjAzICogdG90YWxfc3RlcHMpLCB0b3RhbF9zdGVwcykKICAgIGNvbGxhdGUgPSBNYXNraW5nQ29sbGF0b3IodG9rZW5pemVyLCBtYXhfbGVuPW1heF9sZW4sIHNlZWQ9c2VlZCkKICAgIGRldmljZSA9IG5leHQobW9kZWwucGFyYW1ldGVycygpKS5kZXZpY2UKICAgICMgYXR0ZW50aW9uIG1hc2sgbXVzdCBtYXRjaCB0aGUgbW9kZWwncyBjb21wdXRlIGR0eXBlIChiZjE2IG9uIEdQVSkg4oCUCiAgICAjIFNEUEEgcmVqZWN0cyBtaXNtYXRjaGVkIG1hc2svcXVlcnkgZHR5cGVzIG9uIENVREEgYmFja2VuZHMKICAgIG1hc2tfZHR5cGUgPSBtb2RlbC5nZXRfaW5wdXRfZW1iZWRkaW5ncygpLndlaWdodC5kdHlwZQoKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIHN0ZXAsIG1pY3JvLCB3aW5kb3dfbG9zcywgdDAgPSAwLCAwLCAwLjAsIHRpbWUudGltZSgpCiAgICBtb2RlbC50cmFpbigpCiAgICBkb25lID0gRmFsc2UKICAgIGZvciBlcG9jaCBpbiByYW5nZShlcG9jaHMpOgogICAgICAgIGlmIGRvbmU6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgb3JkZXIgPSBsaXN0KHJhbmdlKGxlbihlbmNvZGVkKSkpCiAgICAgICAgcm5nLnNodWZmbGUob3JkZXIpCiAgICAgICAgZm9yIHN0YXJ0IGluIHJhbmdlKDAsIGxlbihvcmRlciksIGJhdGNoKToKICAgICAgICAgICAgaWR4ID0gb3JkZXJbc3RhcnQ6c3RhcnQgKyBiYXRjaF0KICAgICAgICAgICAgZmVhdHMgPSBjb2xsYXRlKFtlbmNvZGVkW2ldIGZvciBpIGluIGlkeF0pCiAgICAgICAgICAgIGZlYXRzID0ge2s6IHYudG8oZGV2aWNlKSBmb3IgaywgdiBpbiBmZWF0cy5pdGVtcygpfQogICAgICAgICAgICBsb2dpdHMgPSBtb2RlbCgKICAgICAgICAgICAgICAgIGlucHV0X2lkcz1mZWF0c1siaW5wdXRfaWRzIl0sCiAgICAgICAgICAgICAgICBhdHRlbnRpb25fbWFzaz10b19iaWRpcmVjdGlvbmFsX21hc2soCiAgICAgICAgICAgICAgICAgICAgZmVhdHNbImF0dGVudGlvbl9tYXNrIl0sIGR0eXBlPW1hc2tfZHR5cGUpLAogICAgICAgICAgICApLmxvZ2l0cwogICAgICAgICAgICBsb3NzID0gRi5jcm9zc19lbnRyb3B5KAogICAgICAgICAgICAgICAgbG9naXRzLnZpZXcoLTEsIGxvZ2l0cy5zaXplKC0xKSkuZmxvYXQoKSwKICAgICAgICAgICAgICAgIGZlYXRzWyJsYWJlbHMiXS52aWV3KC0xKSwgaWdub3JlX2luZGV4PS0xMDApIC8gYWNjdW0KICAgICAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgICAgIHdpbmRvd19sb3NzICs9IGxvc3MuaXRlbSgpCiAgICAgICAgICAgIG1pY3JvICs9IDEKICAgICAgICAgICAgaWYgbWljcm8gJSBhY2N1bSA9PSAwOgogICAgICAgICAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKAogICAgICAgICAgICAgICAgICAgIChwIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQpLCAxLjApCiAgICAgICAgICAgICAgICBvcHQuc3RlcCgpCiAgICAgICAgICAgICAgICBzY2hlZC5zdGVwKCkKICAgICAgICAgICAgICAgIG9wdC56ZXJvX2dyYWQoc2V0X3RvX25vbmU9VHJ1ZSkKICAgICAgICAgICAgICAgIHN0ZXAgKz0gMQogICAgICAgICAgICAgICAgaWYgc3RlcCAlIGxvZ19ldmVyeSA9PSAwIG9yIHN0ZXAgPT0gMToKICAgICAgICAgICAgICAgICAgICBwcmludChmInN0ZXAge3N0ZXB9L3t0b3RhbF9zdGVwc30gIgogICAgICAgICAgICAgICAgICAgICAgICAgIGYibG9zcyB7d2luZG93X2xvc3MgKiBhY2N1bSAvIG1pY3JvOi40Zn0gIgogICAgICAgICAgICAgICAgICAgICAgICAgIGYiKHt0aW1lLnRpbWUoKSAtIHQwOi4wZn1zKSIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICAgICAgd2luZG93X2xvc3MsIG1pY3JvID0gMC4wLCAwCiAgICAgICAgICAgICAgICBpZiBzdGVwID49IHRvdGFsX3N0ZXBzOgogICAgICAgICAgICAgICAgICAgIGRvbmUgPSBUcnVlCiAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgIG1vZGVsLnNhdmVfcHJldHJhaW5lZChvdXQpCiAgICB0b2tlbml6ZXIuc2F2ZV9wcmV0cmFpbmVkKG91dCkKICAgIHByaW50KGYiYWRhcHRlciBzYXZlZCB0byB7b3V0fSIpCiAgICByZXR1cm4gb3V0CgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCBkZWZhdWx0PSJkYXRhL3NmdF9leGFtcGxlcy5qc29ubCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0ibnBjX2FkYXB0ZXIiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhc2UiLCBkZWZhdWx0PSJEcmVhbS1vcmcvRHJlYW0tdjAtSW5zdHJ1Y3QtN0IiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTEpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTFlLTQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbG9yYS1yIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTYpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2giLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWFjY3VtIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXgtbGVuIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NTEyKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heC1leGFtcGxlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTEyMDAwKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQogICAgdHJhaW4oYmFzZV9tb2RlbD1hcmdzLmJhc2UsIGRhdGE9YXJncy5kYXRhLCBvdXQ9YXJncy5vdXQsIGVwb2Nocz1hcmdzLmVwb2NocywKICAgICAgICAgIGxyPWFyZ3MubHIsIGxvcmFfcj1hcmdzLmxvcmFfciwgYmF0Y2g9YXJncy5iYXRjaCwgYWNjdW09YXJncy5hY2N1bSwKICAgICAgICAgIG1heF9sZW49YXJncy5tYXhfbGVuLCBtYXhfZXhhbXBsZXM9YXJncy5tYXhfZXhhbXBsZXMpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=',
    'npc_engine/demo.py': 'IiIiSW50ZXJhY3RpdmUgdGV4dC1hZHZlbnR1cmUgZGVtbyBmb3IgdGhlIE5QQyBFbmdpbmUuCgpUd28gbW9kZXM6CiAgcHl0aG9uIC1tIG5wY19lbmdpbmUuZGVtbyAgICAgICAgICAgICMgdGFsayB0byB0aGUgTlBDcywgc2VlIGNhbmRpZGF0ZXMgKyBjb25maWRlbmNlcwogIHB5dGhvbiAtbSBucGNfZW5naW5lLmRlbW8gLS1iZW5jaCAgICAjIGxhdGVuY3kgYmVuY2htYXJrOiAyMCB0dXJucywgcmVwb3J0cyBwNTAvcDk1CiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBzdGF0aXN0aWNzCmltcG9ydCBzeXMKCmZyb20gLmVuZ2luZSBpbXBvcnQgTlBDRW5naW5lCmZyb20gLnBlcnNvbmFzIGltcG9ydCBTVEFSVEVSX1BFUlNPTkFTCgpXT1JMRF9TVEFURVMgPSBbCiAgICAiRXZlbmluZy4gVGhlIG1pbGxlcidzIGNhcnQgdGhyZXcgYSB3aGVlbCBvbiB0aGUgbm9ydGggcm9hZDsgYmFuZGl0cyB3ZXJlIHNlZW4gdGhlcmUgbGFzdCB3ZWVrLiIsCiAgICAiTW9ybmluZyBtYXJrZXQgZGF5LiBUaGUgc3F1YXJlIGlzIGJ1c3k7IGEgdHJhdmVsZXIgd2FzIHJvYmJlZCB5ZXN0ZXJkYXkuIiwKICAgICJMYXRlIG5pZ2h0LiBSYWluLiBUaGUgdGF2ZXJuIGlzIG5lYXJseSBlbXB0eTsgYSBzdHJhbmdlciB3YXRjaGVzIGZyb20gdGhlIGNvcm5lci4iLApdCgoKZGVmIGludGVyYWN0aXZlKGVuZ2luZTogTlBDRW5naW5lKToKICAgIHByaW50KCI9PT0gTlBDIEVuZ2luZSDigJQgdGV4dCBhZHZlbnR1cmUgZGVtbyA9PT1cbiIpCiAgICBmb3IgaSwgcCBpbiBlbnVtZXJhdGUoU1RBUlRFUl9QRVJTT05BUyk6CiAgICAgICAgcHJpbnQoZiIgIFt7aX1dIHtwLm5hbWV9IOKAlCB7cC5yb2xlfSIpCiAgICB0cnk6CiAgICAgICAgY2hvaWNlID0gaW50KGlucHV0KCJcblBpY2sgYW4gTlBDICgwLTUpOiAiKS5zdHJpcCgpIG9yICIwIikKICAgICAgICBwZXJzb25hID0gU1RBUlRFUl9QRVJTT05BU1tjaG9pY2UgJSBsZW4oU1RBUlRFUl9QRVJTT05BUyldCiAgICBleGNlcHQgKFZhbHVlRXJyb3IsIEluZGV4RXJyb3IpOgogICAgICAgIHBlcnNvbmEgPSBTVEFSVEVSX1BFUlNPTkFTWzBdCiAgICB3b3JsZCA9IFdPUkxEX1NUQVRFU1swXQogICAgcHJpbnQoZiJcbllvdSBhcHByb2FjaCB7cGVyc29uYS5uYW1lfS4gKHt3b3JsZH0pIikKICAgIHByaW50KCJUeXBlIHRvIHRoZW0uICdxdWl0JyB0byBsZWF2ZSwgJ25wYycgdG8gc3dpdGNoIGNoYXJhY3Rlci5cbiIpCiAgICB3aGlsZSBUcnVlOgogICAgICAgIHRyeToKICAgICAgICAgICAgbGluZSA9IGlucHV0KCJ5b3U+ICIpLnN0cmlwKCkKICAgICAgICBleGNlcHQgKEVPRkVycm9yLCBLZXlib2FyZEludGVycnVwdCk6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgaWYgbm90IGxpbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbGluZS5sb3dlcigpID09ICJxdWl0IjoKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBsaW5lLmxvd2VyKCkgPT0gIm5wYyI6CiAgICAgICAgICAgIHJldHVybiBpbnRlcmFjdGl2ZShlbmdpbmUpCiAgICAgICAgciA9IGVuZ2luZS5yZXNwb25kKHBlcnNvbmEsIHdvcmxkLCBsaW5lKQogICAgICAgIGZsYWcgPSAiIFtGQUxMQkFDS10iIGlmIHIudXNlZF9mYWxsYmFjayBlbHNlICIiCiAgICAgICAgcHJpbnQoZiJcbntwZXJzb25hLm5hbWV9PiB7ci50ZXh0fXtmbGFnfSIpCiAgICAgICAgcHJpbnQoZiIgIChjb25maWRlbmNlIHtyLmNvbmZpZGVuY2U6LjJmfSB8IHtyLmxhdGVuY3lfbXM6LjBmfSBtcyB8ICIKICAgICAgICAgICAgICBmIntsZW4oci5jYW5kaWRhdGVzKX0gdW5pcXVlIGNhbmRpZGF0ZXMpIikKICAgICAgICBpZiBsZW4oci5jYW5kaWRhdGVzKSA+IDE6CiAgICAgICAgICAgIGFsdHMgPSAiLCAiLmpvaW4oZicie3R9IiAoe2M6LjJmfSknIGZvciB0LCBjIGluIHIuY2FuZGlkYXRlc1sxOjRdKQogICAgICAgICAgICBwcmludChmIiAgYWx0ZXJuYXRpdmVzOiB7YWx0c30iKQogICAgICAgIHByaW50KCkKCgpkZWYgYmVuY2htYXJrKGVuZ2luZTogTlBDRW5naW5lKToKICAgIHBlcnNvbmEgPSBTVEFSVEVSX1BFUlNPTkFTWzBdCiAgICB3b3JsZCA9IFdPUkxEX1NUQVRFU1swXQogICAgcHJvYmVzID0gWwogICAgICAgICJBbnkgd29yayBmb3IgbWU/IiwKICAgICAgICAiV2hhdCBoYXBwZW5lZCBvbiB0aGUgbm9ydGggcm9hZD8iLAogICAgICAgICJJIG5lZWQgYSBzd29yZCwgZmFzdC4iLAogICAgICAgICJUZWxsIG1lIGFib3V0IHRoZSBtaWxsZXIuIiwKICAgICAgICAiQXJlIHRoZSBiYW5kaXRzIHN0aWxsIG91dCB0aGVyZT8iLAogICAgXSAqIDQKICAgIGxhdCwgY29uZiwgZmIgPSBbXSwgW10sIDAKICAgIHByaW50KGYicnVubmluZyB7bGVuKHByb2Jlcyl9IHR1cm5zLCA4IGNhbmRpZGF0ZXMgZWFjaC4uLiIpCiAgICBmb3IgcCBpbiBwcm9iZXM6CiAgICAgICAgciA9IGVuZ2luZS5yZXNwb25kKHBlcnNvbmEsIHdvcmxkLCBwKQogICAgICAgIGxhdC5hcHBlbmQoci5sYXRlbmN5X21zKQogICAgICAgIGNvbmYuYXBwZW5kKHIuY29uZmlkZW5jZSkKICAgICAgICBmYiArPSByLnVzZWRfZmFsbGJhY2sKICAgIGxhdC5zb3J0KCkKICAgIHA1MCA9IHN0YXRpc3RpY3MubWVkaWFuKGxhdCkKICAgIHA5NSA9IGxhdFtpbnQoMC45NSAqIGxlbihsYXQpKSAtIDFdCiAgICBwcmludChmIlxucDUwIGxhdGVuY3k6IHtwNTA6LjBmfSBtcyAgIHA5NSBsYXRlbmN5OiB7cDk1Oi4wZn0gbXMiKQogICAgcHJpbnQoZiJtZWFuIGNvbmZpZGVuY2U6IHtzdGF0aXN0aWNzLm1lYW4oY29uZik6LjJmfSAgIGZhbGxiYWNrczoge2ZifS97bGVuKHByb2Jlcyl9IikKICAgIHByaW50KGYidGhyb3VnaHB1dDogezEwMDAgLyBwNTAgKiA4Oi4wZn0gY2FuZGlkYXRlLWxpbmVzL3NlYyAoOCBwZXIgdHVybikiKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1iZW5jaCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PSJEcmVhbS1vcmcvRHJlYW0tdjAtSW5zdHJ1Y3QtN0IiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLThiaXQiLCBkZXN0PSJlaWdodGJpdCIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0ibG9hZCBpbiA4LWJpdCAoZm9yIDE2R0IgR1BVcykiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQogICAgcHJpbnQoImxvYWRpbmcgbW9kZWwgKGZpcnN0IHJ1biBkb3dubG9hZHMgfjE0R0IpLi4uIiwgZmx1c2g9VHJ1ZSkKICAgIGVuZ2luZSA9IE5QQ0VuZ2luZShtb2RlbF9pZD1hcmdzLm1vZGVsLCBsb2FkX2luXzhiaXQ9YXJncy5laWdodGJpdCkKICAgIGlmIGFyZ3MuYmVuY2g6CiAgICAgICAgYmVuY2htYXJrKGVuZ2luZSkKICAgIGVsc2U6CiAgICAgICAgaW50ZXJhY3RpdmUoZW5naW5lKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg==',
    'npc_engine/data/__init__.py': 'IiIiQ29udmVydGVycyBmb3IgbGljZW5zZWQgTlBDIGRpYWxvZ3VlIGRhdGFzZXRzIC0+IHRyYWluaW5nIHRyaXBsZXMuIiIiCg==',
    'npc_engine/data/convert_agentlans.py': 'IiIiTlBDIHRyYWluaW5nLXRyaXBsZSBjb252ZXJ0ZXIuCgphZ2VudGxhbnMvbXVsdGktY2hhcmFjdGVyLWRpYWxvZ3VlIChDQy1CWS00LjApIHJvd3MgbG9vayBsaWtlOgogIHtzZXR0aW5nLCBjaGFyYWN0ZXJzOiB7bmFtZTogZGVzY3JpcHRpb259LCBjb252ZXJzYXRpb246IFt7ZnJvbSwgbWVzc2FnZX1dLAogICAnc2V0dGluZyBhZnRlciBpbnRlcmFjdGlvbid9CgpFYWNoIHR1cm4gaT49MSBiZWNvbWVzIG9uZSB0cmFpbmluZyB0cmlwbGU6CiAgcGVyc29uYSAgICAgID0ge25hbWUsIGRlc2NyaXB0aW9ufSBvZiB0aGUgc3BlYWtlciAodGhlaXIgY2hhcmFjdGVyIGVudHJ5KQogIHdvcmxkX3N0YXRlICA9IHNldHRpbmcKICBjb250ZXh0ICAgICAgPSB0aGUgbGFzdCBgY29udGV4dF90dXJuc2AgcHJpb3IgdHVybnMsIHJlbmRlcmVkICJOYW1lOiBtZXNzYWdlIgogIHBsYXllcl9saW5lICA9IHR1cm4gaS0xJ3MgbWVzc2FnZSAod2hvZXZlciBzcG9rZSBqdXN0IGJlZm9yZSkKICBucGNfbGluZSAgICAgPSB0dXJuIGkncyBtZXNzYWdlICh0aGUgdHJhaW5pbmcgdGFyZ2V0KQoKRmlsdGVyczogdW5rbm93biBzcGVha2VycyBkcm9wcGVkLCBucGNfbGluZSBvdXRzaWRlIFttaW5fd29yZHMsIG1heF93b3Jkc10KZHJvcHBlZCwgZW1wdHkgbWVzc2FnZXMgZHJvcHBlZC4KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGd6aXAKaW1wb3J0IGpzb24KaW1wb3J0IHBhdGhsaWIKaW1wb3J0IHVybGxpYi5yZXF1ZXN0CgpIRl9VUkwgPSAoCiAgICAiaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9kYXRhc2V0cy9hZ2VudGxhbnMvbXVsdGktY2hhcmFjdGVyLWRpYWxvZ3VlIgogICAgIi9yZXNvbHZlL21haW4vdHJhaW4uanNvbmwuZ3oiCikKCgpkZWYgX3dvcmRzKHRleHQ6IHN0cikgLT4gaW50OgogICAgcmV0dXJuIGxlbih0ZXh0LnNwbGl0KCkpCgoKZGVmIGNvbnZlcnRfcm93KHJvdzogZGljdCwgY29udGV4dF90dXJuczogaW50ID0gNiwgbWluX3dvcmRzOiBpbnQgPSAyLAogICAgICAgICAgICAgICAgbWF4X3dvcmRzOiBpbnQgPSAxMjApOgogICAgIiIiUmV0dXJuICh0cmlwbGVzLCBzdGF0cykgZm9yIG9uZSBzY2VuZSByb3cuIiIiCiAgICB0cmlwbGVzLCBza2lwcGVkID0gW10sIHsidW5rbm93bl9zcGVha2VyIjogMCwgImJhZF9sZW5ndGgiOiAwfQogICAgc2V0dGluZyA9IChyb3cuZ2V0KCJzZXR0aW5nIikgb3IgIiIpLnN0cmlwKCkKICAgIGNoYXJhY3RlcnMgPSByb3cuZ2V0KCJjaGFyYWN0ZXJzIikgb3Ige30KICAgIGNvbnZlcnNhdGlvbiA9IHJvdy5nZXQoImNvbnZlcnNhdGlvbiIpIG9yIFtdCiAgICBpZiBub3QgaXNpbnN0YW5jZShjb252ZXJzYXRpb24sIGxpc3QpIG9yIGxlbihjb252ZXJzYXRpb24pIDwgMjoKICAgICAgICByZXR1cm4gdHJpcGxlcywgc2tpcHBlZAogICAgZm9yIGkgaW4gcmFuZ2UoMSwgbGVuKGNvbnZlcnNhdGlvbikpOgogICAgICAgIHR1cm4gPSBjb252ZXJzYXRpb25baV0KICAgICAgICBzcGVha2VyID0gdHVybi5nZXQoImZyb20iKQogICAgICAgIG1lc3NhZ2UgPSAodHVybi5nZXQoIm1lc3NhZ2UiKSBvciAiIikuc3RyaXAoKQogICAgICAgIGlmIG5vdCBtZXNzYWdlIG9yIG5vdCBzcGVha2VyOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGRlc2NyaXB0aW9uID0gY2hhcmFjdGVycy5nZXQoc3BlYWtlcikKICAgICAgICBpZiBub3QgZGVzY3JpcHRpb246CiAgICAgICAgICAgIHNraXBwZWRbInVua25vd25fc3BlYWtlciJdICs9IDEKICAgICAgICAgICAgY29udGludWUKICAgICAgICBudyA9IF93b3JkcyhtZXNzYWdlKQogICAgICAgIGlmIG53IDwgbWluX3dvcmRzIG9yIG53ID4gbWF4X3dvcmRzOgogICAgICAgICAgICBza2lwcGVkWyJiYWRfbGVuZ3RoIl0gKz0gMQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHByZXYgPSBjb252ZXJzYXRpb25baSAtIDFdCiAgICAgICAgcHJpb3IgPSBjb252ZXJzYXRpb25bbWF4KDAsIGkgLSBjb250ZXh0X3R1cm5zKTppXQogICAgICAgIGNvbnRleHQgPSAiXG4iLmpvaW4oCiAgICAgICAgICAgIGYie3QuZ2V0KCdmcm9tJyl9OiB7KHQuZ2V0KCdtZXNzYWdlJykgb3IgJycpLnN0cmlwKCl9IiBmb3IgdCBpbiBwcmlvcgogICAgICAgICkKICAgICAgICB0cmlwbGVzLmFwcGVuZCh7CiAgICAgICAgICAgICJwZXJzb25hIjogeyJuYW1lIjogc3BlYWtlciwgImRlc2NyaXB0aW9uIjogZGVzY3JpcHRpb24uc3RyaXAoKX0sCiAgICAgICAgICAgICJ3b3JsZF9zdGF0ZSI6IHNldHRpbmcsCiAgICAgICAgICAgICJjb250ZXh0IjogY29udGV4dCwKICAgICAgICAgICAgInBsYXllcl9saW5lIjogKHByZXYuZ2V0KCJtZXNzYWdlIikgb3IgIiIpLnN0cmlwKCksCiAgICAgICAgICAgICJucGNfbGluZSI6IG1lc3NhZ2UsCiAgICAgICAgfSkKICAgIHJldHVybiB0cmlwbGVzLCBza2lwcGVkCgoKZGVmIF93cml0YWJsZShzOiBzdHIpIC0+IHN0cjoKICAgICIiIlJlcGxhY2UgbG9uZSBVVEYtMTYgc3Vycm9nYXRlcyAoYnJva2VuIGVtb2ppIGluIHRoZSBzb3VyY2UgZGF0YSkgdGhhdAogICAgc3RyaWN0IHV0Zi04IGNhbm5vdCBlbmNvZGUg4oCUIG9uZSBiYWQgcm93IG11c3Qgbm90IGtpbGwgdGhlIHdob2xlIHJ1bi4iIiIKICAgIHJldHVybiBzLmVuY29kZSgidXRmLTgiLCAicmVwbGFjZSIpLmRlY29kZSgidXRmLTgiKQoKCmRlZiBjb252ZXJ0X2ZpbGUocGF0aDogcGF0aGxpYi5QYXRoLCBvdXRfcGF0aDogcGF0aGxpYi5QYXRoLAogICAgICAgICAgICAgICAgIGNvbnRleHRfdHVybnM9NiwgbWluX3dvcmRzPTIsIG1heF93b3Jkcz0xMjAsIGxpbWl0PU5vbmUpOgogICAgdG90YWxfcm93cywgdG90YWxfdHJpcGxlcyA9IDAsIDAKICAgIHNraXBwZWQgPSB7InVua25vd25fc3BlYWtlciI6IDAsICJiYWRfbGVuZ3RoIjogMH0KICAgIG9wZW5lciA9IGd6aXAub3BlbiBpZiBzdHIocGF0aCkuZW5kc3dpdGgoIi5neiIpIGVsc2Ugb3BlbgogICAgd2l0aCBvcGVuZXIocGF0aCwgInJ0IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZiwgXAogICAgICAgICAgICBvcGVuKG91dF9wYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIG91dDoKICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgICAgIGlmIG5vdCBsaW5lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgdG90YWxfcm93cyArPSAxCiAgICAgICAgICAgIHRyaXBsZXMsIHNrID0gY29udmVydF9yb3coanNvbi5sb2FkcyhsaW5lKSwgY29udGV4dF90dXJucywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtaW5fd29yZHMsIG1heF93b3JkcykKICAgICAgICAgICAgZm9yIHMsIHYgaW4gc2suaXRlbXMoKToKICAgICAgICAgICAgICAgIHNraXBwZWRbc10gKz0gdgogICAgICAgICAgICBmb3IgdCBpbiB0cmlwbGVzOgogICAgICAgICAgICAgICAgb3V0LndyaXRlKF93cml0YWJsZShqc29uLmR1bXBzKHQsIGVuc3VyZV9hc2NpaT1GYWxzZSkpICsgIlxuIikKICAgICAgICAgICAgdG90YWxfdHJpcGxlcyArPSBsZW4odHJpcGxlcykKICAgICAgICAgICAgaWYgbGltaXQgYW5kIHRvdGFsX3Jvd3MgPj0gbGltaXQ6CiAgICAgICAgICAgICAgICBicmVhawogICAgcmV0dXJuIHsicm93cyI6IHRvdGFsX3Jvd3MsICJ0cmlwbGVzIjogdG90YWxfdHJpcGxlcywgInNraXBwZWQiOiBza2lwcGVkfQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhLWRpciIsIGRlZmF1bHQ9ImRhdGEiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxpbWl0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1taW4td29yZHMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1heC13b3JkcyIsIHR5cGU9aW50LCBkZWZhdWx0PTEyMCkKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBkZGlyID0gcGF0aGxpYi5QYXRoKGFyZ3MuZGF0YV9kaXIpCiAgICBkZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHNyYyA9IGRkaXIgLyAiYWdlbnRsYW5zLXRyYWluLmpzb25sLmd6IgogICAgaWYgbm90IHNyYy5leGlzdHMoKToKICAgICAgICBwcmludCgiZG93bmxvYWRpbmcgYWdlbnRsYW5zL211bHRpLWNoYXJhY3Rlci1kaWFsb2d1ZSAoQ0MtQlktNC4wKS4uLiIsIGZsdXNoPVRydWUpCiAgICAgICAgdXJsbGliLnJlcXVlc3QudXJscmV0cmlldmUoSEZfVVJMLCBzcmMpCiAgICBvdXQgPSBkZGlyIC8gIm5wY190cmlwbGVzLmpzb25sIgogICAgc3RhdHMgPSBjb252ZXJ0X2ZpbGUoc3JjLCBvdXQsIG1pbl93b3Jkcz1hcmdzLm1pbl93b3JkcywKICAgICAgICAgICAgICAgICAgICAgICAgIG1heF93b3Jkcz1hcmdzLm1heF93b3JkcywgbGltaXQ9YXJncy5saW1pdCkKICAgIHByaW50KGpzb24uZHVtcHMoc3RhdHMsIGluZGVudD0yKSkKICAgIHByaW50KCJ3cm90ZSIsIG91dCkKICAgICMgc2hvdyBhIHNhbXBsZQogICAgd2l0aCBvcGVuKG91dCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBzYW1wbGUgPSBqc29uLmxvYWRzKGYucmVhZGxpbmUoKSkKICAgIHByaW50KCJcblNBTVBMRSBUUklQTEU6IikKICAgIHByaW50KGpzb24uZHVtcHMoc2FtcGxlLCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlKVs6OTAwXSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==',
    'npc_engine/data/build_sft.py': 'IiIiQnVpbGQgU0ZUIGV4YW1wbGVzIGZyb20gTlBDIHRyaXBsZXMuCgpUcmlwbGUgLT4ge3N5c3RlbSwgdXNlciwgYXNzaXN0YW50fSB3aGVyZToKLSBzeXN0ZW0gPSB0aGUgY2Fub25pY2FsIGVuZ2luZSBwcm9tcHQgKHBlcnNvbmEgY2FyZCArIGhpc3RvcnkgKyBzaXR1YXRpb24KICArIHJ1bGVzKSwgYnVpbHQgYnkgcGVyc29uYXMuc3lzdGVtX3Byb21wdF9mb3Igc28gdHJhaW4gPT0gaW5mZXJlbmNlIGZvcm1hdAotIHVzZXIgPSB0aGUgcGxheWVyIGxpbmUgKHRoZSBsYXN0IHR1cm4gb2YgdGhlIHRyaXBsZSdzIGNvbnRleHQpCi0gYXNzaXN0YW50ID0gdGhlIE5QQydzIGxpbmUKLSBoaXN0b3J5ID0gdGhlIGNvbnRleHQgdHVybnMgQkVGT1JFIHRoZSBwbGF5ZXIgbGluZSwgcmVuZGVyZWQgIk5hbWU6IGxpbmUiCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBwYXRobGliCgpmcm9tIG5wY19lbmdpbmUucGVyc29uYXMgaW1wb3J0IHN5c3RlbV9wcm9tcHRfZm9yCgoKZGVmIHRyaXBsZV90b19leGFtcGxlKHQ6IGRpY3QpIC0+IGRpY3Q6CiAgICBjb250ZXh0X2xpbmVzID0gW2xuIGZvciBsbiBpbiAodC5nZXQoImNvbnRleHQiKSBvciAiIikuc3BsaXQoIlxuIikgaWYgbG4uc3RyaXAoKV0KICAgIGhpc3RvcnkgPSBjb250ZXh0X2xpbmVzWzotMV0gaWYgbGVuKGNvbnRleHRfbGluZXMpID4gMSBlbHNlIE5vbmUKICAgIHN5c3RlbSA9IHN5c3RlbV9wcm9tcHRfZm9yKAogICAgICAgIHRbInBlcnNvbmEiXVsibmFtZSJdLAogICAgICAgIHRbInBlcnNvbmEiXVsiZGVzY3JpcHRpb24iXSwKICAgICAgICB0WyJ3b3JsZF9zdGF0ZSJdLAogICAgICAgIGhpc3Rvcnk9aGlzdG9yeSwKICAgICkKICAgIHJldHVybiB7InN5c3RlbSI6IHN5c3RlbSwgInVzZXIiOiB0WyJwbGF5ZXJfbGluZSJdLCAiYXNzaXN0YW50IjogdFsibnBjX2xpbmUiXX0KCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdHJpcGxlcyIsIGRlZmF1bHQ9ImRhdGEvbnBjX3RyaXBsZXMuanNvbmwiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW91dCIsIGRlZmF1bHQ9ImRhdGEvc2Z0X2V4YW1wbGVzLmpzb25sIikKICAgIGFyZ3MgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBuID0gMAogICAgd2l0aCBvcGVuKGFyZ3MudHJpcGxlcywgZW5jb2Rpbmc9InV0Zi04IikgYXMgZiwgXAogICAgICAgICAgICBvcGVuKGFyZ3Mub3V0LCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIG91dDoKICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgICAgIGlmIG5vdCBsaW5lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgZXggPSB0cmlwbGVfdG9fZXhhbXBsZShqc29uLmxvYWRzKGxpbmUpKQogICAgICAgICAgICBvdXQud3JpdGUoanNvbi5kdW1wcyhleCwgZW5zdXJlX2FzY2lpPUZhbHNlKSArICJcbiIpCiAgICAgICAgICAgIG4gKz0gMQogICAgcHJpbnQoZiJ3cm90ZSB7bn0gU0ZUIGV4YW1wbGVzIHRvIHthcmdzLm91dH0iKQogICAgd2l0aCBvcGVuKGFyZ3Mub3V0LCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHNhbXBsZSA9IGpzb24ubG9hZHMoZi5yZWFkbGluZSgpKQogICAgcHJpbnQoIlxuU0FNUExFOiIpCiAgICBwcmludCgiU1lTVEVNOiIsIHNhbXBsZVsic3lzdGVtIl1bOjQwMF0pCiAgICBwcmludCgiVVNFUjoiLCBzYW1wbGVbInVzZXIiXSkKICAgIHByaW50KCJBU1NJU1RBTlQ6Iiwgc2FtcGxlWyJhc3Npc3RhbnQiXSkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==',
}
for rel, b64 in FILES.items():
    p = pathlib.Path(rel)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
import npc_engine
print("npc_engine version:", npc_engine.__version__, " (must say 0.2.2)")

## 3. Build the training data

In [ ]:
!python -m npc_engine.data.convert_agentlans
!python -m npc_engine.data.build_sft

## 4. Train (LoRA on the 8-bit base)

12,000 examples, 1 epoch, effective batch 16. Watch the loss printouts — it should trend down. If Colab disconnects, rerun from cell 2 (training restarts; data rebuild is fast).

In [ ]:
from npc_engine.finetune import train
train(data='data/sft_examples.jsonl', out='npc_adapter',
      max_examples=12000, epochs=1)

## 5. Base vs fine-tuned, same scenes

Compare the lines: the tuned model's should be cleaner grammatically and stick to 1-2 short lines.

In [ ]:
import gc, torch
from npc_engine import NPCEngine, STARTER_PERSONAS

SCENES = [
    ('Borin Ironfist', 'The market square at noon. A traveler\'s cart has a cracked wheel; bandits were seen on the north road.', 'My cart wheel cracked just outside town. Can you fix it?'),
    ('Mira Thistledown', 'The inn common room at dusk. Market day was busy; a robbery occurred at the miller\'s stall.', 'What\'s the word around town today?'),
    ('Captain Sera Voss', 'The town gate at shift change. A curfew begins at sundown; strangers need a pass.', 'Let me through, I have business inside.'),
]

def run(engine, label):
    print(f'===== {label} =====')
    for name, world, line in SCENES:
        p = STARTER_PERSONAS[name]
        r = engine.respond(p, world, line)
        print(f'{name}: "{r.line}"  (conf {r.confidence:.2f}, {r.latency_ms:.0f} ms)')
    print()

base = NPCEngine(load_in_8bit=True)
run(base, 'BASE MODEL')
del base; gc.collect(); torch.cuda.empty_cache()

tuned = NPCEngine(load_in_8bit=True, adapter_path='npc_adapter')
run(tuned, 'FINE-TUNED')

## 6. Speed check: steps 32 vs 16 (tuned model)

The Phase 0 T4 number was ~58s per turn at 32 steps. Halving the steps should roughly halve that — check the trade against line quality above.

In [ ]:
import time
p = STARTER_PERSONAS['Borin Ironfist']
world = 'The market square at noon. A traveler\'s cart has a cracked wheel; bandits were seen on the north road.'
for steps in (32, 16):
    ts = []
    for _ in range(5):
        t0 = time.perf_counter()
        r = tuned.respond(p, world, 'Can you fix a cracked wheel?', steps=steps)
        ts.append(time.perf_counter() - t0)
    ts.sort()
    print(f'steps={steps}: p50 {ts[2]*1000:.0f} ms/turn, line: "{r.line}"')

## 7. Keep the adapter

The adapter is ~100MB in the `npc_adapter` folder. To keep it past this session, mount Drive and copy it:

```python
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r npc_adapter /content/drive/MyDrive/
```